In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Import your custom classes
from vitsModel.DataLoader import DataLoader
from vitsModel.PrivacyVitsModel import PrivacyVitsModel
from MIAModel.ModelInversionAttack import ModelInversionAttack

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("="*80)
print("📁 STEP 1: LOADING openEDS DATASET")
print("="*80)

image_dir_path = './openEDS/openEDS/train/images'
label_dir_path = './openEDS/openEDS/train/labels'

# Load your dataset
images, labels = DataLoader.load_grayscale_images_label_pairs(
    image_dir_path, label_dir_path,
    step=0, step_size=2500,  # Load 2500 images
    target_size=(224, 224),
    add_noise=False,
    normalize_on_load=False
)

print(f"✅ Loaded {len(images)} images")
print(f"   Image shape: {images.shape}")
print(f"   Label shape: {labels.shape}")

# Normalize images to [0, 1]
if images.max() > 1.0:
    images = images.astype(np.float32) / 255.0
    print(f"✅ Normalized images to [0, 1] range")

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, 
    test_size=0.2, 
    random_state=42
)

print(f"✅ Split data:")
print(f"   Training: {len(X_train)} samples")
print(f"   Testing:  {len(X_test)} samples")

# ============================================================================
# STEP 2: LOAD OR TRAIN MODELS
# ============================================================================
print("\n" + "="*80)
print("🤖 STEP 2: LOADING/TRAINING MODELS")
print("="*80)

privacy_model_class = PrivacyVitsModel()

# Option A: Load pre-trained models (if you have them)
# ----------------------------------------------------------------------------
try:
    print("\n🔄 Attempting to load pre-trained models...")
    
    model_without_privacy = PrivacyVitsModel.load_model(
        'best_enhanced_vit_eye_model_batch_current.h5'
    )
    print("✅ Loaded model WITHOUT privacy layers")
    
    model_with_privacy = PrivacyVitsModel.load_model(
        'best_enhanced_vit_eye_model_batch_current.h5'
    )
    print("✅ Loaded model WITH privacy layers")
    
    models_loaded = True
    
except Exception as e:
    print(f"⚠️  Could not load pre-trained models: {e}")
    print("📝 Will train new models...")
    models_loaded = False

# Option B: Train models if not found
# ----------------------------------------------------------------------------
if not models_loaded:
    print("\n" + "="*80)
    print("🏋️  TRAINING MODELS (this may take a while...)")
    print("="*80)
    
    # Train model WITHOUT privacy layers
    print("\n1️⃣  Training model WITHOUT privacy layers...")
    print("-"*80)
    
    model_without_privacy, callbacks = privacy_model_class.create_and_compile_enhanced_model()
    
    history_no_privacy = privacy_model_class.complete_training_pipeline(
        model_without_privacy,
        X_train, y_train,
        epochs=30,  # Adjust as needed
        batch_size=8,
        test_size=0.2,
        enhanced_augmentation=True,
        verbose_evaluation=True
    )
    
    # Save the model
    model_without_privacy.save('best_model_without_privacy.h5')
    print("✅ Model WITHOUT privacy saved to 'best_model_without_privacy.h5'")
    
    # Train model WITH privacy layers
    print("\n2️⃣  Training model WITH privacy layers...")
    print("-"*80)
    
    model_with_privacy, callbacks = privacy_model_class.create_and_compile_model_with_privacy(
        base_drop_rate=0.15,           # Adaptive patch dropping
        sensitivity_strength=0.5,       # Sensitivity to privacy regions
        dp_clip_norm=1.0,              # Gradient clipping for DP
        anonymization_strength=0.3,     # Feature anonymization
        differential_privacy_noise=0.05 # DP noise level
    )
    
    history_with_privacy = privacy_model_class.complete_training_pipeline(
        model_with_privacy,
        X_train, y_train,
        epochs=30,
        batch_size=8,
        test_size=0.2,
        enhanced_augmentation=True,
        verbose_evaluation=True
    )
    
    # Save the model
    model_with_privacy.save('best_model_with_privacy.h5')
    print("✅ Model WITH privacy saved to 'best_model_with_privacy.h5'")

# ============================================================================
# STEP 3: PREPARE DATA FOR ATTACK
# ============================================================================
print("\n" + "="*80)
print("🎯 STEP 3: PREPARING DATA FOR PRIVACY ATTACKS")
print("="*80)

# Use a subset for attacks (to save time)
num_attack_samples = min(100, len(X_test))
X_test_attack = X_test[:num_attack_samples]
y_test_attack = y_test[:num_attack_samples]

num_train_samples = min(100, len(X_train))
X_train_attack = X_train[:num_train_samples]
y_train_attack = y_train[:num_train_samples]

print(f"✅ Using {num_attack_samples} test samples for attack")
print(f"✅ Using {num_train_samples} train samples for membership inference")

# ============================================================================
# STEP 4: RUN INVERSION ATTACKS
# ============================================================================
print("\n" + "="*80)
print("🔓 STEP 4: RUNNING MODEL INVERSION ATTACKS")
print("="*80)

# Attack each model separately first
print("\n" + "="*80)
print("🎯 ATTACKING MODEL WITHOUT PRIVACY LAYERS")
print("="*80)

attacker_no_privacy = ModelInversionAttack(model_without_privacy)
results_no_privacy = attacker_no_privacy.comprehensive_privacy_test(
    X_test_attack, y_test_attack,
    X_train_attack, y_train_attack,
    num_samples=5
)

print("\n" + "="*80)
print("🎯 ATTACKING MODEL WITH PRIVACY LAYERS")
print("="*80)

attacker_with_privacy = ModelInversionAttack(model_with_privacy)
results_with_privacy = attacker_with_privacy.comprehensive_privacy_test(
    X_test_attack, y_test_attack,
    X_train_attack, y_train_attack,
    num_samples=5
)

# ============================================================================
# STEP 5: COMPARE RESULTS
# ============================================================================
print("\n" + "="*80)
print("📊 STEP 5: COMPARING PRIVACY PROTECTION")
print("="*80)

# Calculate average metrics
def calculate_avg_metrics(results):
    avg_ssim_grad = np.mean([m['gradient_metrics']['SSIM'] for m in results['metrics']])
    avg_ssim_feat = np.mean([m['feature_metrics']['SSIM'] for m in results['metrics']])
    avg_psnr_grad = np.mean([m['gradient_metrics']['PSNR'] for m in results['metrics']])
    avg_psnr_feat = np.mean([m['feature_metrics']['PSNR'] for m in results['metrics']])
    
    return {
        'SSIM_gradient': avg_ssim_grad,
        'SSIM_feature': avg_ssim_feat,
        'PSNR_gradient': avg_psnr_grad,
        'PSNR_feature': avg_psnr_feat,
        'privacy_score': 1.0 - (avg_ssim_grad + avg_ssim_feat) / 2
    }

metrics_no_privacy = calculate_avg_metrics(results_no_privacy)
metrics_with_privacy = calculate_avg_metrics(results_with_privacy)

print("\n📊 COMPARISON TABLE")
print("="*80)
print(f"{'Metric':<25} {'Without Privacy':<20} {'With Privacy':<20} {'Improvement'}")
print("-"*80)

for key in ['SSIM_gradient', 'SSIM_feature', 'PSNR_gradient', 'PSNR_feature']:
    val_no = metrics_no_privacy[key]
    val_with = metrics_with_privacy[key]
    
    # For SSIM and PSNR, lower is better for privacy
    if 'SSIM' in key:
        improvement = ((val_no - val_with) / val_no * 100) if val_no > 0 else 0
        symbol = '↓'
    else:  # PSNR
        improvement = ((val_no - val_with) / val_no * 100) if val_no > 0 else 0
        symbol = '↓'
    
    print(f"{key:<25} {val_no:<20.4f} {val_with:<20.4f} {improvement:+.1f}% {symbol}")

print("-"*80)
privacy_no = metrics_no_privacy['privacy_score'] * 100
privacy_with = metrics_with_privacy['privacy_score'] * 100
improvement = privacy_with - privacy_no

print(f"{'Privacy Score':<25} {privacy_no:<20.1f}% {privacy_with:<20.1f}% {improvement:+.1f}%")
print("="*80)

# Membership inference comparison
if 'membership_inference' in results_no_privacy and 'membership_inference' in results_with_privacy:
    print("\n📊 MEMBERSHIP INFERENCE ATTACK RESULTS")
    print("="*80)
    
    mem_no = results_no_privacy['membership_inference']
    mem_with = results_with_privacy['membership_inference']
    
    print(f"{'Model':<25} {'Attack Accuracy':<20} {'Privacy Protection'}")
    print("-"*80)
    print(f"{'Without Privacy':<25} {mem_no['overall_accuracy']*100:<20.2f}% {'Lower is better'}")
    print(f"{'With Privacy':<25} {mem_with['overall_accuracy']*100:<20.2f}% {'Lower is better'}")
    print("="*80)

# ============================================================================
# STEP 6: VISUALIZE COMPARISON
# ============================================================================
print("\n" + "="*80)
print("🎨 STEP 6: CREATING VISUALIZATIONS")
print("="*80)

# Visualize attacks on multiple samples
num_viz_samples = min(3, len(results_no_privacy['gradient_reconstructions']))

for i in range(num_viz_samples):
    print(f"\n📸 Creating visualization for sample {i+1}...")
    
    sample_img = X_test_attack[i:i+1]
    
    reconstructions = [
        results_no_privacy['gradient_reconstructions'][i],
        results_with_privacy['gradient_reconstructions'][i],
        results_no_privacy['feature_reconstructions'][i],
        results_with_privacy['feature_reconstructions'][i]
    ]
    
    names = [
        'No Privacy\n(Gradient Attack)',
        'With Privacy\n(Gradient Attack)',
        'No Privacy\n(Feature Attack)',
        'With Privacy\n(Feature Attack)'
    ]
    
    attacker_no_privacy.visualize_attack_results(
        sample_img[0], reconstructions, names,
        save_path=f'privacy_attack_sample_{i+1}.png'
    )

print("\n✅ All visualizations saved!")

# ============================================================================
# STEP 7: GENERATE SUMMARY REPORT
# ============================================================================
print("\n" + "="*80)
print("📋 FINAL PRIVACY PROTECTION REPORT")
print("="*80)

print(f"\n🔒 Model WITHOUT Privacy Layers:")
print(f"   Privacy Score:        {privacy_no:.1f}%")
print(f"   Avg SSIM (Gradient):  {metrics_no_privacy['SSIM_gradient']:.4f}")
print(f"   Avg SSIM (Feature):   {metrics_no_privacy['SSIM_feature']:.4f}")

print(f"\n🛡️  Model WITH Privacy Layers:")
print(f"   Privacy Score:        {privacy_with:.1f}%")
print(f"   Avg SSIM (Gradient):  {metrics_with_privacy['SSIM_gradient']:.4f}")
print(f"   Avg SSIM (Feature):   {metrics_with_privacy['SSIM_feature']:.4f}")

print(f"\n✨ Overall Improvement:")
print(f"   Privacy Score:        {improvement:+.1f}%")

if improvement > 20:
    print(f"   Status:               ✅ SIGNIFICANT IMPROVEMENT!")
elif improvement > 10:
    print(f"   Status:               ⚠️  MODERATE IMPROVEMENT")
else:
    print(f"   Status:               ❌ MINIMAL IMPROVEMENT")

print("\n" + "="*80)
print("🎉 PRIVACY ATTACK ANALYSIS COMPLETED!")
print("="*80)

print(f"\n📁 Results saved:")
print(f"   - Models: best_model_without_privacy.h5, best_model_with_privacy.h5")
print(f"   - Visualizations: privacy_attack_sample_*.png")
print("\n💡 Lower SSIM and reconstruction quality = Better privacy protection!")

2025-10-06 15:05:11.210981: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 15:05:11.305821: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 15:05:13.036808: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


ImportError: cannot import name 'ModelInversionAttack' from 'MIAModel.mia' (/root/projects/GazeEstimation/MIAModel/mia.py)